# Revised Transformer training and benchmark evaluation

This notebook documents a revised training procedure for subsequent-shot and
same-team-goal prediction. It reads completed experiment artifacts without fitting
models, selecting hyperparameters or rescoring test data. Run cells in order with
the project kernel. Installation and experiment commands are in the README and
[experiment specification](../docs/transformer-retraining.md).

The first experiment and its outputs remain in notebook 10. This run changes the
training procedure, retaining its five input features and causal architecture.


## 1. Experiment verification

The loader resolves paths relative to the repository, verifies artifact and
source hashes, and reads the saved protocol and selected configurations.
Completion requires all six refits to be frozen before test loading.


In [1]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p/'data/sources.json').is_file() and (p/'pyproject.toml').is_file()), None)
if project_root is None:
    raise RuntimeError('Launch from the repository root or notebooks folder.')
experiment = project_root/'artifacts/transformer-retrained-2026'
if not (experiment/'summary.json').is_file():
    raise RuntimeError('Complete python -m afl_opposition.transformer_retraining first; use --resume if interrupted.')
summary = json.loads((experiment/'summary.json').read_text(encoding='utf-8'))
protocol = json.loads((experiment/'protocol.json').read_text(encoding='utf-8'))
def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()
for name,expected in summary['artifact_sha256'].items():
    assert digest(experiment/name) == expected,name
for name,expected in protocol['source_sha256'].items():
    assert digest(project_root/'src/afl_opposition'/name) == expected,name
assert summary['all_refits_frozen_before_test_loading']
assert summary['models_unchanged_during_evaluation']
targets = protocol['targets']
winners = {t:json.loads((experiment/t/'selection.json').read_text()) for t in targets}
refits = {t:json.loads((experiment/t/'refit.json').read_text()) for t in targets}
print('Experiment:',experiment.relative_to(project_root).as_posix())
print('Verified artifacts:',len(summary['artifact_sha256']))
print('PyTorch:',protocol['torch_version'],'Device:',summary['device'])
print('Primary seed:',protocol['primary_seed'],'All seeds:',protocol['seeds'])


Experiment: artifacts/transformer-retrained-2026
Verified artifacts: 2016
PyTorch: 2.11.0+cu128 Device: cuda
Primary seed: 2026 All seeds: [2026, 2027, 2028]


## 2. Hyperparameter search coverage

The 64 configurations form a bounded conditional search. Every listed parameter
value appears; valid head counts divide model width and give 8–64 dimensions per
head. Coverage of values does not imply exhaustive coverage of combinations.
Four anchor configurations include the earlier winners and compact alternatives.


In [2]:
candidates = pd.DataFrame(protocol['candidates'])
assert len(candidates.drop_duplicates()) == 64
assert (candidates.d_model % candidates.num_heads).eq(0).all()
assert (candidates.d_model/candidates.num_heads).between(8,64).all()
coverage = pd.DataFrame([dict(setting=c,values=sorted(candidates[c].unique().tolist()),
    counts=candidates[c].value_counts().sort_index().to_dict()) for c in candidates])
display(coverage)
print('First four anchor configurations:')
display(candidates.head(4))


,setting,values,counts
0,num_layers,"[1, 2, 3]","{1: 22, 2: 21, 3: 21}"
1,d_model,"[32, 64, 128]","{32: 21, 64: 21, 128: 22}"
2,num_heads,"[1, 2, 4, 8]","{1: 6, 2: 27, 4: 20, 8: 11}"
3,ff_multiplier,"[2, 4]","{2: 32, 4: 32}"
4,embedding_dim,"[4, 8, 16]","{4: 22, 8: 21, 16: 21}"
5,continuous_dim,"[8, 16, 32]","{8: 21, 16: 24, 32: 19}"
6,dropout,"[0.0, 0.1, 0.2, 0.3]","{0.0: 18, 0.1: 17, 0.2: 15, 0.3: 14}"
7,learning_rate,"[0.0001, 0.0003, 0.001, 0.003]","{0.0001: 14, 0.0003: 16, 0.001: 17, 0.003: 17}"
8,lambda_l2,"[0.0, 1e-07, 1e-06, 1e-05, 0.0001]","{0.0: 12, 1e-07: 13, 1e-06: 17, 1e-05: 11, 0.0..."
9,warmup_fraction,"[0.0, 0.05, 0.1]","{0.0: 21, 0.05: 24, 0.1: 19}"


First four anchor configurations:


,num_layers,d_model,num_heads,ff_multiplier,embedding_dim,continuous_dim,dropout,learning_rate,lambda_l2,warmup_fraction
0,3,32,4,4,4,16,0.1,0.0030,0.000001,0.05
1,3,128,4,2,4,16,0.0,0.0010,0.000001,0.05
2,1,32,2,2,8,16,0.0,0.0010,0.000001,0.05
3,2,64,4,2,8,16,0.1,0.0003,0.000001,0.05


## 3. Batching, learning-rate schedule and search budgets

An effective batch contains up to 1,024 randomly selected possessions. Length
sorting occurs within that batch before division into microbatches of at most
512. Gradients accumulate with normalisation by the effective batch count.
Each effective batch applies the L2 penalty, gradient clipping and optimiser
update once.

Each screen completes its own 300-update decay schedule. Promoted configurations
restart from scratch for 3,200 updates. The best three get two further seeds; final
selection minimises mean 2025 BCE at a common checkpoint across all three seeds.
Refitting replays that many updates of the same full schedule on 2023–2025.


In [3]:
display(pd.DataFrame([
    dict(stage='Screen',configurations=64,seeds=1,updates_per_fit=300),
    dict(stage='Full runs (fresh)',configurations=8,seeds=1,updates_per_fit=3200),
    dict(stage='Additional finalist seeds',configurations=3,seeds=2,updates_per_fit=3200),
]))
print('Effective batch:',protocol['effective_batch_size'],'Microbatch:',protocol['microbatch_size'])
print('Schedule:',protocol['schedule'])
print('Optimiser:',protocol['optimiser'])
print('Selection checkpoint interval:',protocol['evaluate_every'])
print('Fixed classification threshold:',protocol['threshold'])


,stage,configurations,seeds,updates_per_fit
0,Screen,64,1,300
1,Full runs (fresh),8,1,3200
2,Additional finalist seeds,3,2,3200


Effective batch: 1024 Microbatch: 512
Schedule: linear warmup then cosine decay to 5% of peak; refit replays selected update prefix
Optimiser: Adam, zero weight_decay, explicit L2 on matrix weights
Selection checkpoint interval: 100
Fixed classification threshold: 0.5


## 4. Matched training diagnostic

These four 600-update runs use the same anchored architecture, seed and effective
batch size to compare batching and schedule. The table reports final checkpoints;
complete learning curves remain in the artifacts. Training audit metrics use 4,096 fixed
random fitting possessions with dropout disabled. Validation uses the full 2025
population. The diagnostic does not select the final model or explain every change.


In [4]:
for target in targets:
    frame = pd.read_parquet(experiment/target/'diagnostics/comparison.parquet')
    print(target)
    display(frame.loc[frame.step.eq(600),['batching','schedule','step','training_audit_bce',
        'training_audit_calibration_gap','validation_bce','validation_calibration_gap']]
        .style.format(precision=6))


shot


,batching,schedule,step,training_audit_bce,training_audit_calibration_gap,validation_bce,validation_calibration_gap
5,bucketed,constant,600,0.399766,-0.008189,0.407985,-0.014131
11,bucketed,cosine,600,0.389763,-0.007706,0.398639,-0.013400
17,random,constant,600,0.380011,0.005985,0.387674,0.000477
23,random,cosine,600,0.379814,0.004400,0.387830,-0.000986


goal


,batching,schedule,step,training_audit_bce,training_audit_calibration_gap,validation_bce,validation_calibration_gap
5,bucketed,constant,600,0.261797,0.008883,0.285070,-0.004465
11,bucketed,cosine,600,0.256534,0.008684,0.281083,-0.004987
17,random,constant,600,0.255367,0.020459,0.277798,0.007365
23,random,cosine,600,0.254125,0.014200,0.277702,0.001183


## 5. Candidate promotion and full-budget validation

Promotion takes the top four screen results plus protected low-rate, compact, deep
and wide candidates. This reduces, but does not eliminate, short-screen bias.
All eight promoted candidates finish full training. Their best single-seed validation
checkpoints determine which three receive additional seeds.


In [5]:
for target in targets:
    screen = pd.read_parquet(experiment/target/'screen.parquet')
    full = pd.read_parquet(experiment/target/'full_search.parquet')
    assert len(screen)==64 and screen.promoted.sum()==8 and full.finalist.sum()==3
    print(target,'— promoted screen configurations')
    display(screen.loc[screen.promoted,['trial','num_layers','d_model','learning_rate',
        'dropout','lambda_l2','validation_bce','best_step']].sort_values('validation_bce').style.format(precision=6))
    print('Full 3,200-update runs (seed 2026):')
    display(full[['trial','parameters','validation_bce','best_step','finalist']]
        .sort_values('validation_bce').style.format({'validation_bce':'{:.6f}'}))


shot — promoted screen configurations


,trial,num_layers,d_model,learning_rate,dropout,lambda_l2,validation_bce,best_step
53,53,1,128,0.003000,0.000000,0.000000,0.386855,300
32,32,3,64,0.003000,0.000000,0.000001,0.388440,300
48,48,3,128,0.003000,0.000000,0.000000,0.388880,300
21,21,1,128,0.003000,0.100000,0.000000,0.389084,300
1,1,3,128,0.001000,0.000000,0.000001,0.389349,300
18,18,3,128,0.003000,0.200000,0.000010,0.390087,300
26,26,1,32,0.003000,0.000000,0.000001,0.390976,300
24,24,1,64,0.000300,0.000000,0.000000,0.398322,300


Full 3,200-update runs (seed 2026):


,trial,parameters,validation_bce,best_step,finalist
0,53,139521,0.382460,2900,True
1,32,102089,0.383100,2900,True
3,21,138385,0.383131,2800,True
2,48,597529,0.383178,2900,False
5,26,9153,0.383186,2800,False
6,1,400649,0.383506,2100,False
7,18,400649,0.384435,3200,False
4,24,52593,0.384845,2900,False


goal — promoted screen configurations


,trial,num_layers,d_model,learning_rate,dropout,lambda_l2,validation_bce,best_step
53,53,1,128,0.003000,0.000000,0.000000,0.277280,300
32,32,3,64,0.003000,0.000000,0.000001,0.277702,300
21,21,1,128,0.003000,0.100000,0.000000,0.278183,300
52,52,3,64,0.003000,0.200000,0.000010,0.278645,300
48,48,3,128,0.003000,0.000000,0.000000,0.278682,300
31,31,2,128,0.003000,0.300000,0.000001,0.278746,300
0,0,3,32,0.003000,0.100000,0.000001,0.278779,300
24,24,1,64,0.000300,0.000000,0.000000,0.282320,300


Full 3,200-update runs (seed 2026):


,trial,parameters,validation_bce,best_step,finalist
0,53,139521,0.275678,2600,True
1,32,102089,0.275828,2900,True
6,48,597529,0.275902,2200,True
5,0,39017,0.276001,2900,False
2,21,138385,0.276033,2900,False
3,52,104129,0.276064,3200,False
4,24,52593,0.276518,2900,False
7,31,402449,0.276745,3200,False


## 6. Three-seed selection and selected architecture

Selection minimises mean validation BCE across three seeds at a common update
for each finalist, then chooses the lowest mean across finalists. Individual
seed performance does not determine the winner. The standard deviation
describes seed sensitivity, not a confidence interval. Trial numbers below
use the saved zero-based identifiers.


In [6]:
selected = []
for target,winner in winners.items():
    selection = pd.read_parquet(experiment/target/'seed_selection.parquet')
    eligible = selection.loc[selection.step.ge(200)]
    assert eligible.loc[eligible.selected,'validation_bce'].iloc[0] == eligible.validation_bce.min()
    print(target,'— best common checkpoint for each finalist')
    display(eligible.sort_values(['validation_bce','parameters','trial','step'])
        .drop_duplicates('trial')[['trial','step','validation_bce','seed_std','selected']]
        .style.format({'validation_bce':'{:.6f}','seed_std':'{:.6f}'}))
    selected.append(dict(target=target,**winner['config'],updates=winner['steps'],
        parameters=winner['parameters'],mean_validation_bce=winner['mean_validation_bce']))
display(pd.DataFrame(selected).set_index('target').T.style.format('{:.6g}'))


shot — best common checkpoint for each finalist


,trial,step,validation_bce,seed_std,selected
95,53,3200,0.382854,0.000187,True
59,32,2800,0.383242,0.000343,False
27,21,2800,0.383322,0.000426,False


goal — best common checkpoint for each finalist


,trial,step,validation_bce,seed_std,selected
95,53,3200,0.275915,0.000140,True
25,32,2600,0.275984,0.000127,False
53,48,2200,0.276130,0.000248,False


target,shot,goal
num_layers,1,1
d_model,128,128
ff_multiplier,2,2
embedding_dim,16,16
continuous_dim,32,32
dropout,0,0
learning_rate,0.003,0.003
lambda_l2,1e-07,1e-07
warmup_fraction,0.1,0.1
num_heads,2,2


## 7. Evaluation-mode learning curves

The learning-curve sample includes every 400 updates, the selected checkpoint
and the final checkpoint for the winning architecture. Training audit and
validation losses both use fixed weights and dropout disabled. The audit is a sample of training possessions, so it is not
the exact full training loss. Online loss additionally involves changing weights
and active dropout. Complete curves are saved for all seeds and finalists.


In [7]:
for target,winner in winners.items():
    curves = pd.read_parquet(experiment/target/'seed_learning_curves.parquet')
    rows = curves.loc[curves.trial.eq(winner['trial']) &
        (curves.step.mod(400).eq(0) | curves.step.eq(winner['steps']))].copy()
    rows['selected_update'] = rows.step.eq(winner['steps'])
    print(target,'— selected at budget boundary:',winner['steps']==protocol['full_steps'])
    display(rows[['seed','step','learning_rate','online_training_bce','training_audit_bce',
        'validation_bce','validation_calibration_gap','selected_update']].style.format(precision=6))


shot — selected at budget boundary: True


,seed,step,learning_rate,online_training_bce,training_audit_bce,validation_bce,validation_calibration_gap,selected_update
3,2026,400,0.002995,0.377439,0.378935,0.387039,-0.008922,False
7,2026,800,0.002810,0.374476,0.378901,0.388789,-0.030217,False
11,2026,1200,0.002393,0.373446,0.378174,0.385812,-0.009741,False
15,2026,1600,0.001823,0.372923,0.374988,0.383785,-0.010154,False
19,2026,2000,0.001207,0.371286,0.375142,0.383470,-0.004631,False
23,2026,2400,0.000659,0.371156,0.375048,0.383146,0.003552,False
27,2026,2800,0.000284,0.369674,0.373623,0.382525,0.002394,False
31,2026,3200,0.000150,0.371705,0.373238,0.382637,-0.006192,True
35,2027,400,0.002995,0.377264,0.386458,0.393282,-0.031198,False
39,2027,800,0.002810,0.374752,0.379867,0.388509,-0.013246,False


goal — selected at budget boundary: True


,seed,step,learning_rate,online_training_bce,training_audit_bce,validation_bce,validation_calibration_gap,selected_update
3,2026,400,0.002995,0.271363,0.254176,0.278057,0.001736,False
7,2026,800,0.002810,0.269984,0.251326,0.278542,-0.016505,False
11,2026,1200,0.002393,0.269896,0.252494,0.276989,0.001560,False
15,2026,1600,0.001823,0.268671,0.251881,0.276961,-0.006186,False
19,2026,2000,0.001207,0.268326,0.250899,0.276445,-0.005487,False
23,2026,2400,0.000659,0.268482,0.250344,0.275889,-0.002083,False
27,2026,2800,0.000284,0.266105,0.250242,0.275787,-0.001734,False
31,2026,3200,0.000150,0.267071,0.250013,0.275753,-0.002372,True
35,2027,400,0.002995,0.273172,0.252513,0.278114,-0.007823,False
39,2027,800,0.002810,0.271281,0.251930,0.276710,-0.001810,False


## 8. Fresh refits and pre-test checks

Refits use all 2023–2025 possessions, new initial weights and a newly fitted encoder.
The update budget and learning-rate schedule come from validation; no test outcomes
enter fitting. A development-only sanity gate requires training BCE below the
constant predictor and an absolute calibration gap no greater than 0.03. These
full-population checks are more exact than the sampled training audit above.


In [8]:
rows=[]
for target,refit in refits.items():
    assert refit['steps']==winners[target]['steps']
    assert refit['schedule_horizon']==protocol['full_steps']
    diagnostics = pd.read_parquet(experiment/target/'refit_training_diagnostics.parquet')
    assert diagnostics.calibration_gap.abs().le(protocol['refit_calibration_gap_limit']).all()
    rows.append(diagnostics.assign(target=target,updates=refit['steps'],
        approximate_passes=refit['steps']/refit['updates_per_epoch']))
display(pd.concat(rows,ignore_index=True)[['target','seed','updates','approximate_passes',
    'possessions','bce','mean_probability','observed_rate','calibration_gap']].style.format(precision=6))


,target,seed,updates,approximate_passes,possessions,bce,mean_probability,observed_rate,calibration_gap
0,shot,2026,3200,21.192053,153822,0.373560,0.197557,0.199692,-0.002135
1,shot,2027,3200,21.192053,153822,0.373712,0.201010,0.199692,0.001318
2,shot,2028,3200,21.192053,153822,0.373917,0.192944,0.199692,-0.006748
3,goal,2026,3200,21.192053,153822,0.269708,0.095102,0.098555,-0.003453
4,goal,2027,3200,21.192053,153822,0.269754,0.098077,0.098555,-0.000479
5,goal,2028,3200,21.192053,153822,0.270004,0.094189,0.098555,-0.004367


## 9. Shared 2026 benchmark comparison

Lower BCE/Brier and higher AP are better. These metrics give each possession total
weight one, regardless of its number of event prefixes. Classification uses threshold
0.5. The primary revised model uses predeclared seed 2026. The separately labelled
ensemble averages three probabilities and has additional inference cost.

The 2026 season has already been observed in previous experiments. It remains a
shared benchmark; these results are not an independent fresh-holdout confirmation.


In [9]:
comparisons = {t:pd.read_parquet(experiment/t/'comparison/metrics.parquet') for t in targets}
metrics = pd.concat([frame.assign(target=t) for t,frame in comparisons.items()],ignore_index=True)
names = ['log_loss','brier_score','average_precision','accuracy','precision','recall',
         'specificity','f1','balanced_accuracy']
display(metrics[['target','model',*names]].style.format({c:'{:.6f}' for c in names},na_rep='undefined'))
deltas=[]
for target,frame in comparisons.items():
    indexed=frame.set_index('model')
    for baseline in ['seq2event_transformer','lstm_history','xgboost']:
        deltas.append(dict(target=target,compared_with=baseline,**{
            m:indexed.loc['transformer_retrained',m]-indexed.loc[baseline,m]
            for m in ['log_loss','brier_score','average_precision']}))
print('Primary revised model minus baseline; negative BCE/Brier and positive AP favour the revised model:')
display(pd.DataFrame(deltas).style.format({m:'{:+.6f}' for m in ['log_loss','brier_score','average_precision']}))


,target,model,log_loss,brier_score,average_precision,accuracy,precision,recall,specificity,f1,balanced_accuracy
0,shot,constant,0.523087,0.169815,0.216341,0.783659,undefined,0.000000,1.000000,0.000000,0.500000
1,shot,logistic,0.423148,0.134152,0.555929,0.815618,0.672837,0.287535,0.961403,0.402895,0.624469
2,shot,xgboost,0.398320,0.125400,0.612536,0.829024,0.756987,0.308838,0.972630,0.438695,0.640734
3,shot,lstm_no_history,0.401857,0.126606,0.606363,0.827301,0.724644,0.325362,0.965869,0.449087,0.645616
4,shot,lstm_history,0.394988,0.124298,0.618982,0.829832,0.716298,0.353392,0.961360,0.473285,0.657376
5,shot,seq2event_transformer,0.417787,0.133067,0.610934,0.814198,0.586665,0.477774,0.907072,0.526650,0.692423
6,shot,transformer_retrained,0.393005,0.123556,0.623397,0.830915,0.758239,0.320676,0.971774,0.450728,0.646225
7,shot,transformer_retrained_ensemble,0.392694,0.123430,0.624178,0.831123,0.758996,0.321469,0.971820,0.451646,0.646645
8,goal,constant,0.337951,0.094666,0.105809,0.894191,undefined,0.000000,1.000000,0.000000,0.500000
9,goal,logistic,0.297569,0.085786,0.284618,0.894596,0.589756,0.012575,0.998965,0.024625,0.505770


Primary revised model minus baseline; negative BCE/Brier and positive AP favour the revised model:


,target,compared_with,log_loss,brier_score,average_precision
0,shot,seq2event_transformer,-0.024782,-0.009511,+0.012463
1,shot,lstm_history,-0.001983,-0.000742,+0.004414
2,shot,xgboost,-0.005315,-0.001844,+0.010861
3,goal,seq2event_transformer,-0.004174,-0.001123,+0.010885
4,goal,lstm_history,-0.000488,-0.000191,+0.004212
5,goal,xgboost,-0.002373,-0.000577,+0.008782


## 10. Raw and weighted confusion matrices

Rows represent actual outcome and columns predicted outcome: `[[TN, FP], [FN, TP]]`.
Raw counts are event prefixes. Weighted counts sum inverse-length prefix weights,
so their total is the number of possessions. Both use the unchanged 0.5 threshold.
The primary model is shown; ensemble matrices are in the documentation and artifacts.


In [10]:
for target in targets:
    confusion = pd.read_parquet(experiment/target/'comparison/confusion.parquet')
    for weighting in ['unweighted','possession_weighted']:
        row = confusion.loc[confusion.model.eq('transformer_retrained') & confusion.weighting.eq(weighting)].iloc[0]
        matrix = pd.DataFrame([[row.tn,row.fp],[row.fn,row.tp]],
            index=[f'Actual no {target}',f'Actual {target}'],columns=[f'Predicted no {target}',f'Predicted {target}'])
        print(target,weighting,'— threshold 0.5')
        display(matrix.style.format('{:.0f}' if weighting=='unweighted' else '{:.2f}'))


shot unweighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,201918,6877
Actual shot,64200,17914


shot possession_weighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,39016.71,1133.29
Actual shot,7529.63,3554.37


goal unweighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,248260,1265
Actual goal,38789,2595


goal possession_weighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,45555.49,257.51
Actual goal,4882.52,538.48


## 11. Seed variability and calibration

The evaluation reports every seed. The comparison between mean predicted
probability and observed outcome rate measures overall calibration; a small
overall gap does not establish calibration within every probability band.
The final table gives those bands for the primary model. No recalibration is fitted here.


In [11]:
for target in targets:
    seed_metrics=pd.read_parquet(experiment/target/'test_seed_metrics.parquet')
    print(target,'— individual seeds')
    display(seed_metrics[['seed','log_loss','brier_score','average_precision']].style.format(precision=6))
    print('Mean and sample standard deviation across three seeds; these are not confidence intervals:')
    display(seed_metrics[['log_loss','brier_score','average_precision']].agg(['mean','std']).style.format(precision=6))
    rates=[]
    sources = [('first_transformer',project_root/'artifacts/transformer-2026'/target/'history/test/predictions.parquet'),
               ('retrained_primary',experiment/target/'test/seed-2026/predictions.parquet'),
               ('retrained_ensemble',experiment/target/'test/ensemble/predictions.parquet')]
    for name,path in sources:
        p=pd.read_parquet(path)
        weight=p.sample_weight
        mean=(p[target+'_probability']*weight).sum()/weight.sum()
        observed=(p.target*weight).sum()/weight.sum()
        rates.append(dict(model=name,mean_probability=mean,observed_rate=observed,gap=mean-observed))
    display(pd.DataFrame(rates).style.format(precision=6))
    calibration=pd.read_parquet(experiment/target/'comparison/calibration.parquet')
    display(calibration.loc[calibration.model.eq('transformer_retrained'),
        ['bin_lower','bin_upper','examples','weight_sum','mean_prediction',f'observed_{target}_rate']]
        .style.format(precision=4))


shot

 — individual seeds


,seed,log_loss,brier_score,average_precision
0,2026,0.393005,0.123556,0.623397
1,2027,0.392685,0.123426,0.623514
2,2028,0.393780,0.123784,0.623099


Mean and sample standard deviation across three seeds; these are not confidence intervals:


,log_loss,brier_score,average_precision
mean,0.393157,0.123589,0.623336
std,0.000563,0.000181,0.000214


...

,seed,log_loss,brier_score,average_precision
0,2026,0.285207,0.081911,0.351444
1,2027,0.285039,0.081868,0.352274
2,2028,0.285477,0.081924,0.352007


Mean and sample standard deviation across three seeds; these are not confidence intervals:


,log_loss,brier_score,average_precision
mean,0.285241,0.081901,0.351909
std,0.000221,0.000029,0.000424


,model,mean_probability,observed_rate,gap
0,first_transformer,0.107635,0.105809,0.001826
1,retrained_primary,0.098348,0.105809,-0.007461
2,retrained_ensemble,0.099041,0.105809,-0.006767


,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_goal_rate
47,0.0000,0.1000,178307,34213.2686,0.0437,0.0497
48,0.1000,0.2000,74399,11149.0634,0.1394,0.1504
49,0.2000,0.3000,24375,3405.9051,0.2419,0.2471
50,0.3000,0.4000,7422,1156.9146,0.3414,0.3714
51,0.4000,0.5000,2546,512.8557,0.4417,0.4534
52,0.5000,0.6000,1372,267.1278,0.5477,0.5347
53,0.6000,0.7000,864,193.4323,0.6469,0.6411
54,0.7000,0.8000,705,155.2342,0.7522,0.7369
55,0.8000,0.9000,746,152.9417,0.8499,0.8652
56,0.9000,1.0000,173,27.2565,0.9128,0.9143


## Interpretation limits

For **shots**, the revised primary model's BCE changes by **-0.024782** relative to the first Transformer and **-0.001983** relative to the history LSTM. The lowest single-model test BCE is from `transformer_retrained`. Negative changes are improvements. At threshold 0.5, the revised model has higher precision, lower recall and lower F1 than the history LSTM.

For **goals**, the revised primary model's BCE changes by **-0.004174** relative to the first Transformer and **-0.000488** relative to the history LSTM. The lowest single-model test BCE is from `transformer_retrained`. Negative changes are improvements. At threshold 0.5, the revised model has higher precision, lower recall and lower F1 than the history LSTM.

The broader search and repeated seeds give stronger evidence about these training
settings, but do not guarantee optimal hyperparameters or statistical superiority.
Final differences combine batching, schedule and search changes. Earlier models
were not retuned under identical budgets. The ensemble is a separate, more costly
prediction option. Independent temporal evaluation would strengthen deployment claims.

Public possession boundaries and labels remain inferred, one 2024 fixture is absent,
and the features do not include defensive tracking. Most sequences are short. More
model capacity cannot compensate for missing information. Separate shot and goal
models do not enforce probability ordering.
